In [6]:
import os
import numpy as np
from xml.dom import minidom
import torch
from torch.utils.data import Dataset
from torchvision import tv_tensors
from PIL import Image
import cv2

In [7]:
directory = os.getcwd()

In [8]:
class BaseballVideos(Dataset):
    def __init__(self, root=None, transforms=None):
        self.root = root if root is not None else os.getcwd()
        self.transforms = transforms

        # Gather video and annotation files
        self.vids = sorted([f for f in os.listdir(self.root) if f.endswith(".mov")])
        self.notes = sorted([f for f in os.listdir(self.root) if f.endswith(".xml")])

        # Build a list of (video_path, xml_path, frame_number)
        self.frame_index = []
        for vid, note in zip(self.vids, self.notes):
            vid_path = os.path.join(self.root, vid)
            note_path = os.path.join(self.root, note)
            cap = cv.VideoCapture(vid_path)
            frame_count = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
            for f in range(frame_count):
                self.frame_index.append((vid_path, note_path, f))
            cap.release()

    def __len__(self):
        return len(self.frame_index)

    def __getitem__(self, idx):
        vid_path, note_path, frame_num = self.frame_index[idx]

        # Load the frame on-the-fly
        cap = cv.VideoCapture(vid_path)
        cap.set(cv.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        cap.release()
        if not ret:
            raise RuntimeError(f"Could not read frame {frame_num} from {vid_path}")

        # Convert frame to PIL Image (cv2 loads as BGR, PIL expects RGB)
        frame_rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
        img = Image.fromarray(frame_rgb)

        # Parse XML for this frame
        note = minidom.parse(note_path)
        frame_i = [j for j in note.getElementsByTagName("box") if int(j.attributes['frame'].value) == frame_num]
        boxes, labels, areas, movings = [], [], [], []

        canvas_size = [img.height, img.width]
        for j in frame_i:
            moving = j.getElementsByTagName('attribute')[0].firstChild.data == 'true'
            xtl = float(j.attributes['xtl'].value)
            ytl = float(j.attributes['ytl'].value)
            xbr = float(j.attributes['xbr'].value)
            ybr = float(j.attributes['ybr'].value)
            box = (xtl, ytl, xbr, ybr)
            label = 'baseball'
            area = (xbr - xtl) * (ybr - ytl)
            boxes.append(box)
            labels.append(label)
            areas.append(area)
            movings.append(moving)

        target = {
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=canvas_size),
            "labels": labels,
            "area": areas,
            "moving": movings
        }

        if self.transforms is not None:
            img = self.transforms(img)  # Only image is transformed here

        return img, target

In [9]:
# pretrained model
import torchvision.models as models
import torch.nn as nn

class BaseballTrackerPretrained(nn.Module):
    def __init__(self, num_outputs=4):
        super().__init__()
        # Load pretrained ResNet18
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Remove the final classification layer
        self.backbone = nn.Sequential(*list(self.backbone.children())[:-1])  # Output: (batch, 512, 1, 1)
        self.flatten = nn.Flatten()
        self.fc_bbox = nn.Linear(512, num_outputs)  # For bounding box regression
        self.fc_move = nn.Linear(512, 1)            # For movement classification

    def forward(self, x):
        x = self.backbone(x)
        x = self.flatten(x)
        bbox = self.fc_bbox(x)
        is_moving = torch.sigmoid(self.fc_move(x))
        return bbox, is_moving

In [10]:
import torchvision.transforms as T

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

### Test Model

In [58]:
test_directory = os.path.join(directory,'test')
test_file = os.path.join(test_directory,'IMG_8946_souleymane.mov')


In [59]:
output_file = os.path.join(test_directory,'test2.mp4')


In [13]:
model = BaseballTrackerPretrained()
model.load_state_dict(torch.load('baseball_tracker_pretrained_epoch20.pth'))
model.eval()

BaseballTrackerPretrained(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_

In [14]:
import cv2
import numpy as np
import torch

# ✅ must match training
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1) Define/import the SAME model class you trained ---
# from your_model_file import BaseballTrackerPretrained
# (or paste the class definition above this)

# def load_model(weights_path: str):
#     model = BaseballTrackerPretrained().to(DEVICE)
#     state = torch.load(weights_path, map_location=DEVICE)
#     model.load_state_dict(state)
#     model.eval()
#     return model

from PIL import Image

def preprocess_frame_bgr(frame_bgr):
    """
    OpenCV BGR uint8 -> transformed tensor [3,224,224]
    """
    # BGR -> RGB
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # NumPy -> PIL
    frame_pil = Image.fromarray(frame_rgb)

    # Apply SAME transform as training
    x = transform(frame_pil)   # [3,224,224], normalized

    return x


@torch.no_grad()
def predict_video(
    video_path: str,
    model: torch.nn.Module,
    batch_size: int = 16,
    img_size=None,  # e.g. (224,224) if you trained resized; else None
):

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    preds = []  # list of dicts: {"frame_idx": i, "bbox": [x1,y1,x2,y2], "move_prob": p}

    frames_buf = []
    idx_buf = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        x = preprocess_frame_bgr(frame)
        # print(x.shape)      # torch.Size([3,224,224])
        # print(x.min(), x.max())
        frames_buf.append(x)
        idx_buf.append(frame_idx)

        # run a batch
        if len(frames_buf) == batch_size:
            batch = torch.stack(frames_buf, dim=0).to(DEVICE)  # [B,3,H,W]
            pred_bbox, pred_move = model(batch)               # [B,4], [B,1]

            pred_bbox = pred_bbox.detach().cpu().numpy()
            pred_move = pred_move.detach().cpu().numpy().reshape(-1)

            for i, fi in enumerate(idx_buf):
                preds.append({
                    "frame_idx": fi,
                    "bbox": pred_bbox[i].tolist(),
                    "move_prob": float(pred_move[i]),
                })

            frames_buf, idx_buf = [], []

        frame_idx += 1

    # flush leftovers
    if len(frames_buf) > 0:
        batch = torch.stack(frames_buf, dim=0).to(DEVICE)
        pred_bbox, pred_move = model(batch)

        pred_bbox = pred_bbox.detach().cpu().numpy()
        pred_move = pred_move.detach().cpu().numpy().reshape(-1)

        for i, fi in enumerate(idx_buf):
            preds.append({
                "frame_idx": fi,
                "bbox": pred_bbox[i].tolist(),
                "move_prob": float(pred_move[i]),
            })

    cap.release()
    return preds

In [15]:
if __name__ == "__main__":
    video_path = test_file
    preds = predict_video(video_path, model, batch_size=16)

    # Print a few predictions
    for p in preds[:5]:
        print(p)
    print("Total frames predicted:", len(preds))

{'frame_idx': 0, 'bbox': [965.6726684570312, 939.5873413085938, 961.25390625, 956.0843505859375], 'move_prob': 0.7234067320823669}
{'frame_idx': 1, 'bbox': [965.7843627929688, 939.6945190429688, 961.3636474609375, 956.194580078125], 'move_prob': 0.724482536315918}
{'frame_idx': 2, 'bbox': [966.2279052734375, 940.1268310546875, 961.8048095703125, 956.632080078125], 'move_prob': 0.7224511504173279}
{'frame_idx': 3, 'bbox': [968.5921630859375, 942.42724609375, 964.1593017578125, 958.9737548828125], 'move_prob': 0.723517894744873}
{'frame_idx': 4, 'bbox': [974.090576171875, 947.7862548828125, 969.6334838867188, 964.4256591796875], 'move_prob': 0.722543478012085}
Total frames predicted: 55


In [78]:
import numpy as np
import cv2

def draw_box(frame, bbox, move_prob, thr=0.5):
    """
    frame: HxWx3 uint8 (BGR) numpy array
    bbox:  [x1,y1,x2,y2] in pixel coords (can be torch/numpy/float)
    """

    # Ensure frame is contiguous uint8 for OpenCV
    if frame.dtype != np.uint8:
        frame = np.clip(frame, 0, 255).astype(np.uint8)
    frame = np.ascontiguousarray(frame)

    H, W = frame.shape[:2]

    # Flatten bbox and convert to python floats first
    b = np.array(bbox, dtype=np.float32).reshape(-1)
    if b.size != 4:
        return frame  # nothing to draw

    x1, y1, x2, y2 = b.tolist()

    # If model sometimes outputs reversed corners, fix it
    if x2 < x1: x1, x2 = x2, x1
    if y2 < y1: y1, y2 = y2, y1

    #Clamp to image bounds (note: max index is W-1/H-1)
    x1 = int(max(0, min(W - 1, round(x1))))
    y1 = int(max(0, min(H - 1, round(y1))))
    x2 = int(max(0, min(W - 1, round(x2))))
    y2 = int(max(0, min(H - 1, round(y2))))

    # Avoid zero/negative-size rectangles
    if x2 <= x1 or y2 <= y1:
        return frame

    color = (0, 255, 0) if float(move_prob) > thr else (0, 0, 255)

    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

    label = f"move={float(move_prob):.2f}"
    cv2.putText(frame, label, (x1, max(0, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)

    return frame


In [55]:
def draw_boxes(image, boxes, move_prob):
    """
    Draw bounding boxes on an image.
    """
    H, W = image.shape[:2]
    
    label = f"move={float(move_prob):.2f}"
    
    #for (xtl, ytl, xbr, ybr) in boxes:
    x1 = int(round(boxes[0]))
    y1 = int(round(boxes[1]))
    x2 = int(round(boxes[2]))
    y2 = int(round(boxes[3]))
    
    scale = 6  # 50% bigger

    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    w  = (x2 - x1) * scale
    h  = (y2 - y1) * scale

    x1b = int(max(0, cx - w / 2))
    y1b = int(max(0, cy - h / 2))
    x2b = int(min(W - 1, cx + w / 2))
    y2b = int(min(H - 1, cy + h / 2))

    cv2.rectangle(image, (x1b, y1b), (x2b, y2b), (0, 255, 0), 4)
    cv2.putText(
        image,
        str(label),
        (x1, max(0, y1 - 5)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0),
        2,
        cv2.LINE_AA,
    )
    return image

In [56]:
def annotate_video(input_path, output_path, preds, thr=0.5):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {input_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (W, H))

    # Build lookup by frame index
    pred_map = {p["frame_idx"]: p for p in preds}

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx in pred_map:
            p = pred_map[frame_idx]
            # frame = draw_box(
            #     frame,
            #     bbox=p["bbox"],
            #     move_prob=p["move_prob"],
            #     thr=thr,
            # )
            frame = draw_boxes(
                frame,
                boxes=p["bbox"],
                move_prob=p["move_prob"]
            )

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    print(f"Saved annotated video → {output_path}")

In [60]:
if __name__ == "__main__":
    video_path = test_file
    output_path = output_file
    preds = predict_video(video_path, model, batch_size=16)
    print("Total frames predicted:", len(preds))
    
    annotate_video(video_path, output_file, preds)
    print(f"Saved {output_path}")

Total frames predicted: 94
Saved annotated video → /Users/josephcoldanghise/Desktop/Baseball Project/baseball_project/test/test2.mp4
Saved /Users/josephcoldanghise/Desktop/Baseball Project/baseball_project/test/test2.mp4
